# Loading Data


In [1]:
import pandas as pd
import ast
import numpy as np

In [63]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/dnb-handle-data")

print("Path to dataset files:", path)

100%|██████████| 319k/319k [00:00<00:00, 658kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/dnb-handle-data/versions/2


In [64]:
mbm_bat = pd.read_csv(path + '/ganjoo_ipl_mbm_bat.csv')
playing_xi = pd.read_csv(path + '/playing_xi.csv')
player_info = pd.read_csv(path + '/ganjoo_player_id_mapping.csv')

# Handling DNB Values

In [65]:
# Update team names in team1_name and team2_name columns
playing_xi['team1_name'] = playing_xi['team1_name'].replace({'Delhi Daredevils': 'Delhi Capitals',
                                                               'Kings XI Punjab': 'Punjab Kings',
                                                            'Rising Pune Supergiants': 'Rising Pune Supergiant'})
playing_xi['team2_name'] = playing_xi['team2_name'].replace({'Delhi Daredevils': 'Delhi Capitals',
                                                               'Kings XI Punjab': 'Punjab Kings',
                                                            'Rising Pune Supergiants': 'Rising Pune Supergiant'})

In [66]:
mbm_bat['batter_id'] = mbm_bat['batter_id'].astype(str)

def parse_playing_xi(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

# Apply the conversion on both team1 and team2 playing XI columns
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(parse_playing_xi)
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(parse_playing_xi)

# Create player mapping dictionary (key: player_id as string, value: player_name)
player_map = dict(zip(player_info['player_id'].astype(str), player_info['player_name']))

In [67]:
dnb_rows = []

# Group mbm_bat by match_id and innings (since a match can have multiple innings)
for (match_id, inns), group in mbm_bat.groupby(['match_id', 'inns']):
    # Get common details from the group (assumed to be the same for all rows in the group)
    date = group['date'].iloc[0]
    bat_team = group['bat_team'].iloc[0]
    bowl_team = group['bowl_team'].iloc[0]
    venue = group['venue'].iloc[0]

    # Determine the set of batter IDs (as strings) who batted in this innings
    batted_ids = set(group['batter_id'].unique())

    # Lookup the corresponding playing_xi row for this match_id
    xi_row = playing_xi[playing_xi['match_id'] == match_id]
    if xi_row.empty:
        print(f"Warning: Match ID {match_id} not found in playing XI data.")
        continue

    row = xi_row.iloc[0]
    # Determine which playing XI list corresponds to the batting team
    if row['team1_name'] == bat_team:
        full_xi = row['team1_playing_xi']
    elif row['team2_name'] == bat_team:
        full_xi = row['team2_playing_xi']
    else:
        print(f"Warning: Batting team {bat_team} not found in playing XI teams for match {match_id}.")
        continue

    # Convert all player IDs in the playing XI list to strings
    full_xi = [str(pid) for pid in full_xi]

    # Identify players in the playing XI who did not bat (i.e. DNB)
    dnb_ids = [pid for pid in full_xi if pid not in batted_ids]

    # For each DNB player, create a new row with default values
    for pid in dnb_ids:
        batter_name = player_map.get(pid, "")
        new_row = {
            'date': date,
            'match_id': match_id,
            'bat_team': bat_team,
            'bowl_team': bowl_team,
            'inns': inns,
            'venue': venue,
            'batter_id': pid,
            'batter': batter_name,
            'runs': 0,
            'balls': 0,
            'fours': 0,
            'sixes': 0,
            'sr': 0,
            'isOut': "DNB",
            'wicketType': "DNB",
            'bowler_id': "",
            'bowler': ""
        }
        dnb_rows.append(new_row)

In [68]:
dnb_df = pd.DataFrame(dnb_rows)

combined_df = pd.concat([mbm_bat, dnb_df], ignore_index=True)

combined_df['batter'] = combined_df.apply(
    lambda row: row['batter'] if pd.notnull(row['batter']) and row['batter'] != ""
    else player_map.get(row['batter_id'], ""), axis=1
)

combined_df['is_DNB'] = combined_df['isOut'].apply(lambda x: 1 if x == "DNB" else 0)
combined_df = combined_df.sort_values(by=['match_id', 'inns', 'is_DNB']).reset_index(drop=True)
combined_df.drop(columns=['is_DNB'], inplace=True)

In [69]:
combined_df['bowler_id'] = combined_df['bowler_id'].fillna("")
combined_df['bowler_id'] = combined_df['bowler'].fillna("")

In [70]:
combined_df.to_csv('ganjoo_ipl_mbm_bat_final.csv', index=False)